# ARIMAX Pipeline

Пайплайн для прогнозирования СКР/ОПЖ с использованием экзогенных переменных.

- **Таргет**: СКР или ОПЖ
- **Индекс**: Регион (для группировки временных рядов)
- **Экзогенные переменные**: все остальные признаки из файла
- **Тестовый год**: 2023

In [45]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

## Конфигурация

In [46]:
__MODE__ = 0
FILE_PATH = ("СКР.xlsx", "ОПЖ.xlsx")[__MODE__]
TARGET_COL = ("СКР", "ОПЖ")[__MODE__]
REGION_COL = "Регион"
YEAR_COL = "Год"
TEST_YEAR = 2023
P_RANGE = range(0, 3)
D_RANGE = range(0, 2)
Q_RANGE = range(0, 3)
MIN_YEARS = 5

## Вспомогательные функции

In [47]:
def clean_numeric(series):
    """Очистка числовых значений от пробелов и запятых"""
    return (
        series.astype(str)
        .str.replace('\xa0', '', regex=False)
        .str.replace(' ', '')
        .str.replace(',', '.')
        .astype(float)
    )


def standardize_region_names(df, region_col='Регион'):
    """Стандартизация названий регионов"""
    df_clean = df.copy()
    
    replacements = {
        'Ненецкий авт.округ': 'Ненецкий автономный округ',
        'Hенецкий авт.округ': 'Ненецкий автономный округ',
        '  Ненецкий автономный округ': 'Ненецкий автономный округ',
        'Ямало-Ненецкий авт.округ': 'Ямало-Ненецкий автономный округ',
        'Ямало-Hенецкий авт.округ': 'Ямало-Ненецкий автономный округ',
        '  Ямало-Ненецкий автономный округ': 'Ямало-Ненецкий автономный округ',
        'Ханты-Мансийский авт.округ-Югра': 'Ханты-Мансийский автономный округ - Югра',
        '  Ханты-Мансийский автономный округ - Югра': 'Ханты-Мансийский автономный округ - Югра',
        'Республика Татарстан(Татарстан)': 'Республика Татарстан',
        'Чувашская Республика(Чувашия)': 'Чувашская Республика',
        'Республика Северная Осетия- Алания': 'Республика Северная Осетия-Алания',
        'Oмская область': 'Омская область',
        'Hижегородская область': 'Нижегородская область',
        'г. Севастополь': 'г.Севастополь',
        'г.Москва': 'г.Москва',
        'г.Санкт-Петербург': 'г.Санкт-Петербург',
        'Чукотский авт.округ': 'Чукотский автономный округ',
    }
    
    df_clean[region_col] = df_clean[region_col].replace(replacements)
    df_clean[region_col] = df_clean[region_col].str.strip()
    return df_clean


def prepare_exog_data(df, target_col, region_col, year_col):
    """Подготовка экзогенных переменных"""
    # Определяем экзогенные колонки (все кроме таргета, региона и года)
    exog_cols = [col for col in df.columns if col not in [target_col, region_col, year_col]]
    
    print(f"\nНайдены экзогенные переменные ({len(exog_cols)}):")
    for col in exog_cols:
        print(f"  - {col}")
    
    return exog_cols

## Загрузка и подготовка данных

In [55]:
# Загрузка данных
print(f"Загрузка данных из {FILE_PATH}...")
df = pd.read_excel(FILE_PATH)

print(f"Размер датасета: {df.shape}")
print(f"\nКолонки: {list(df.columns)}")

# Стандартизация названий регионов
print("\nСтандартизация названий регионов...")
df = standardize_region_names(df, REGION_COL)

# Очистка таргета
df[TARGET_COL] = clean_numeric(df[TARGET_COL])

# Преобразование года
df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors='coerce').astype(int)

# Определение экзогенных переменных
exog_cols = prepare_exog_data(df, TARGET_COL, REGION_COL, YEAR_COL)

# Очистка экзогенных переменных (преобразуем в числа)
for col in exog_cols:
    try:
        df[col] = clean_numeric(df[col])
    except:
        print(f"Предупреждение: не удалось преобразовать {col} в числовой тип")

# Удаление строк с пропусками в экзогенных переменных
df = df.dropna(subset=exog_cols + [TARGET_COL])

print(f"\nРазмер после очистки: {df.shape}")
print(f"Годы в данных: {sorted(df[YEAR_COL].unique())}")
print(f"Количество регионов: {df[REGION_COL].nunique()}")

Загрузка данных из СКР.xlsx...
Размер датасета: (850, 14)

Колонки: ['Регион', 'Год', 'СКР', 'Численность населения', 'Число родившихся', 'Браков', 'Разводов', 'Введено в действие общей площади жилых домов на 1000 человек населения', 'Кол-во преступлений', 'Уровень безработицы', 'Уровень бедности', 'Величина прожиточного минимума', 'Валовой региональный продукт на душу населения (ОКВЭД 2)', 'Средняя ЗП']

Стандартизация названий регионов...

Найдены экзогенные переменные (11):
  - Численность населения
  - Число родившихся
  - Браков
  - Разводов
  - Введено в действие общей площади жилых домов на 1000 человек населения
  - Кол-во преступлений
  - Уровень безработицы
  - Уровень бедности
  - Величина прожиточного минимума
  - Валовой региональный продукт на душу населения (ОКВЭД 2)
  - Средняя ЗП

Размер после очистки: (850, 14)
Годы в данных: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022),

## ARIMAX моделирование

In [56]:
def fit_arimax_for_region(region_data, target_col, exog_cols, test_year, 
                          p_range, d_range, q_range):
    """
    Обучение ARIMAX модели для одного региона.
    
    Returns:
        dict с результатами или None при ошибке
    """
    # Разделение на train/test
    train_data = region_data[region_data[YEAR_COL] < test_year].sort_values(YEAR_COL)
    test_data = region_data[region_data[YEAR_COL] == test_year]
    
    if len(test_data) == 0:
        return None, "нет данных за тестовый год"
    
    if len(train_data) < MIN_YEARS:
        return None, f"мало обучающих данных ({len(train_data)} лет)"
    
    # Подготовка данных
    y_train = train_data[target_col].values
    y_test = test_data[target_col].values[0]
    
    X_train = train_data[exog_cols].values
    X_test = test_data[exog_cols].values
    
    # Нормализация экзогенных переменных
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    best_forecast = np.nan
    best_aic = np.inf
    best_order = None
    
    # Перебор параметров ARIMAX
    for p in p_range:
        for d in d_range:
            for q in q_range:
                try:
                    model = ARIMA(
                        endog=y_train,
                        exog=X_train_scaled,
                        order=(p, d, q)
                    )
                    fitted = model.fit()
                    forecast = fitted.forecast(steps=1, exog=X_test_scaled)[0]
                    
                    if np.isfinite(forecast) and fitted.aic < best_aic:
                        best_aic = fitted.aic
                        best_forecast = forecast
                        best_order = (p, d, q)
                except:
                    continue
    
    # Fallback на ARIMA без экзогенных переменных
    if pd.isna(best_forecast):
        try:
            model = ARIMA(y_train, order=(1, 1, 1))
            fitted = model.fit()
            best_forecast = fitted.forecast(steps=1)[0]
            best_order = (1, 1, 1)
        except:
            return None, "не удалось построить модель"
    
    return {
        'forecast': best_forecast,
        'actual': y_test,
        'order': best_order,
        'aic': best_aic if best_aic != np.inf else None
    }, None

In [ ]:
# Основной цикл по регионам
regions = df[REGION_COL].unique()
results = []

print(f"\n{'='*70}")
print(f"ARIMAX прогнозирование {TARGET_COL} на {TEST_YEAR} год")
print(f"{'='*70}\n")

for region in sorted(regions):
    region_data = df[df[REGION_COL] == region]
    
    result, error = fit_arimax_for_region(
        region_data=region_data,
        target_col=TARGET_COL,
        exog_cols=exog_cols,
        test_year=TEST_YEAR,
        p_range=P_RANGE,
        d_range=D_RANGE,
        q_range=Q_RANGE
    )
    
    if error:
        print(f"⚠ {region}: {error}")
        continue
    
    forecast = result['forecast']
    actual = result['actual']
    order = result['order']
    
    abs_error = abs(forecast - actual)
    rmse = np.sqrt(mean_squared_error([actual], [forecast]))
    mae = mean_absolute_error([actual], [forecast])
    mape = abs_error / actual * 100 if actual != 0 else np.nan
    
    results.append({
        'Регион': region,
        f'predictions': round(forecast, 4),
        f'{("СКР", "ОПЖ")[__MODE__]}': round(actual, 4),
        'Абсолютная_ошибка': round(abs_error, 4),
        'RMSE': round(rmse, 4),
        'MAE': round(mae, 4),
        'MAPE_%': round(mape, 2),
        'ARIMA_order': str(order)
    })
    
    print(f"✓ {region}: прогноз={forecast:.4f}, факт={actual:.4f}, "
          f"ошибка={abs_error:.4f}, order={order}")


ARIMAX прогнозирование СКР на 2023 год

✓ Алтайский край: прогноз=1.2379, факт=1.3050, ошибка=0.0671, order=(0, 0, 0)
✓ Амурская область: прогноз=1.5115, факт=1.4900, ошибка=0.0215, order=(1, 0, 0)
✓ Архангельская область: прогноз=1.4803, факт=1.4590, ошибка=0.0213, order=(0, 0, 0)
✓ Астраханская область: прогноз=1.3505, факт=1.6350, ошибка=0.2845, order=(0, 0, 0)
✓ Белгородская область: прогноз=1.1230, факт=1.1230, ошибка=0.0000, order=(0, 0, 0)
✓ Брянская область: прогноз=1.0876, факт=1.1900, ошибка=0.1024, order=(0, 0, 0)
✓ Владимирская область: прогноз=1.1733, факт=1.1470, ошибка=0.0263, order=(0, 0, 0)
✓ Волгоградская область: прогноз=1.1126, факт=1.1190, ошибка=0.0064, order=(0, 0, 0)
✓ Вологодская область: прогноз=1.4479, факт=1.3910, ошибка=0.0569, order=(1, 0, 0)
✓ Воронежская область: прогноз=1.2827, факт=1.2220, ошибка=0.0607, order=(0, 0, 0)
✓ Еврейская автономная область: прогноз=1.7119, факт=1.5590, ошибка=0.1529, order=(0, 0, 0)
✓ Забайкальский край: прогноз=1.5296, фак

## Итоговые метрики и сохранение результатов

In [54]:
if results:
    results_df = pd.DataFrame(results)
    
    # Расчёт общих метрик
    forecast_col = "predictions"
    actual_col = ("СКР", "ОПЖ")[__MODE__]
    
    overall_rmse = np.sqrt(mean_squared_error(
        results_df[actual_col], 
        results_df[forecast_col]
    ))
    overall_mae = mean_absolute_error(
        results_df[actual_col], 
        results_df[forecast_col]
    )
    
    print(f"\n{'='*70}")
    print(f"ИТОГОВЫЕ МЕТРИКИ ПО {TARGET_COL}")
    print(f"{'='*70}")
    print(f"Количество регионов: {len(results)}")
    print(f"Общий RMSE: {overall_rmse:.4f}")
    print(f"Общий MAE:  {overall_mae:.4f}")
    print(f"{'='*70}")
    
    # Статистика по ошибкам
    print(f"\nСтатистика абсолютных ошибок:")
    print(f"  Мин:    {results_df['Абсолютная_ошибка'].min():.4f}")
    print(f"  Макс:   {results_df['Абсолютная_ошибка'].max():.4f}")
    print(f"  Медиана: {results_df['Абсолютная_ошибка'].median():.4f}")
    print(f"  Среднее: {results_df['Абсолютная_ошибка'].mean():.4f}")
    
    # Топ-5 лучших и худших прогнозов
    print(f"\nТоп-5 лучших прогнозов (минимальная ошибка):")
    print(results_df.nsmallest(5, 'Абсолютная_ошибка')[['Регион', forecast_col, actual_col, 'Абсолютная_ошибка']].to_string(index=False))
    
    print(f"\nТоп-5 худших прогнозов (максимальная ошибка):")
    print(results_df.nlargest(5, 'Абсолютная_ошибка')[['Регион', forecast_col, actual_col, 'Абсолютная_ошибка']].to_string(index=False))
    
    # Сохранение результатов
    output_file = ("predictions_afr.xlsx", "predictions_ele.xlsx")[__MODE__]
    results_df.to_excel(output_file, index=False)
    print(f"\n✓ Результаты сохранены в: {output_file}")
    
    # Для Google Colab - раскомментируйте следующие строки:
    # from google.colab import files
    # files.download(output_file)
else:
    print("❌ Нет данных для обработки.")


ИТОГОВЫЕ МЕТРИКИ ПО СКР
Количество регионов: 85
Общий RMSE: 0.4445
Общий MAE:  0.2686

Статистика абсолютных ошибок:
  Мин:    0.0073
  Макс:   2.2269
  Медиана: 0.1336
  Среднее: 0.2686

Топ-5 лучших прогнозов (минимальная ошибка):
              Регион  predictions   СКР  Абсолютная_ошибка
   г.Санкт-Петербург       1.2643 1.257             0.0073
     Республика Крым       1.4097 1.419             0.0093
Республика Татарстан       1.4641 1.454             0.0101
 Челябинская область       1.4604 1.471             0.0106
Новгородская область       1.2703 1.257             0.0133

Топ-5 худших прогнозов (максимальная ошибка):
               Регион  predictions   СКР  Абсолютная_ошибка
    Республика Адыгея      -0.8809 1.346             2.2269
 Республика Ингушетия       3.5733 1.806             1.7673
  Республика Калмыкия       0.4690 1.443             0.9740
Волгоградская область       0.1616 1.119             0.9574
 Оренбургская область       0.6420 1.502             0.8600

✓ Ре